## Global Imports & Setup

In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report, roc_auc_score,
    ConfusionMatrixDisplay, confusion_matrix, f1_score
)
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.ensemble import RandomForestClassifier
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

np.random.seed(42)

COLS = ['age','sex','cp','trestbps','chol','fbs','restecg',
        'thalach','exang','oldpeak','slope','ca','thal','target']

print('All imports successful.')

All imports successful.


## Required Preprocessing [12 Marks]

### Pre-1: Load CSV, Confirm Shape, Print First 5 Rows & Dtypes

In [3]:
# Load the data — using the uploaded file path (update if running locally)
DATA_PATH = '/content/processed.cleveland.data'  # place file in same directory
df = pd.read_csv(DATA_PATH, header=None, names=COLS)

print('Shape:', df.shape)
print('\nFirst 5 rows:')
display(df.head())
print('\nData types:')
print(df.dtypes)

Shape: (303, 14)

First 5 rows:


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0



Data types:
age         float64
sex         float64
cp          float64
trestbps    float64
chol        float64
fbs         float64
restecg     float64
thalach     float64
exang       float64
oldpeak     float64
slope       float64
ca           object
thal         object
target        int64
dtype: object


## Pre-2: Handle Missing Values ('?' → NaN), Report & Drop

In [4]:
# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

# Convert all columns to numeric
df = df.apply(pd.to_numeric, errors='coerce')

print('Missing values per column:')
missing = df.isnull().sum()
print(missing[missing > 0])
print(f'\nRows with missing values: {df.isnull().any(axis=1).sum()}')

df.dropna(inplace=True)
print(f'Retained rows after dropping NaN: {len(df)}')

# Binarise target: 0 = no disease, 1 = disease present
df['target'] = (df['target'] > 0).astype(int)

Missing values per column:
ca      4
thal    2
dtype: int64

Rows with missing values: 6
Retained rows after dropping NaN: 297


## Pre-3: Class Distribution & SMOTE Decision

In [5]:
counts = df['target'].value_counts()
pcts   = df['target'].value_counts(normalize=True) * 100
dist_df = pd.DataFrame({'Count': counts, 'Percent': pcts.round(1)})
dist_df.index = ['No Disease (0)', 'Disease (1)']
print('Class Distribution:')
display(dist_df)

Class Distribution:


,Count,Percent
No Disease (0),160,53.9
Disease (1),137,46.1


**Interpretation:**

The dataset shows a mild imbalance (~54% no disease vs ~46% disease present).
The ratio is not severe enough to necessitate SMOTE, however we apply it on
the training split only to give the minority class a slight boost and reduce
false-negative risk critical in cardiac screening.

### Pre-4 & Pre-5: Encoding, Scaling & Stratified 80/20 Split

In [ ]:
CAT_COLS  = ['cp', 'restecg', 'slope', 'thal']
CONT_COLS = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']

# One-hot encode categoricals
df_enc = pd.get_dummies(df, columns=CAT_COLS, drop_first=False)

X = df_enc.drop('target', axis=1)
y = df_enc['target']

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Fit scaler on training set only
scaler = StandardScaler()
cont_in_X = [c for c in CONT_COLS if c in X_train.columns]
X_train[cont_in_X] = scaler.fit_transform(X_train[cont_in_X])
X_test[cont_in_X]  = scaler.transform(X_test[cont_in_X])

print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')
print(f'Train class balance: {y_train.value_counts().to_dict()}')

# Apply SMOTE on training split only
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)
print(f'After SMOTE — Train size: {X_train_sm.shape}, Balance: {pd.Series(y_train_sm).value_counts().to_dict()}')

### Pre-6: Correlation Heatmap of Original Numeric Features

In [ ]:
numeric_orig = df[['age','trestbps','chol','thalach','oldpeak','ca','target']]
corr = numeric_orig.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap of Numeric Features', fontsize=13)
plt.tight_layout()
plt.show()

# Top 3 correlated pairs (excluding self-correlation)
corr_unstacked = corr.where(~mask).stack().abs().sort_values(ascending=False)
print('Top 3 feature pairs by absolute correlation:')
print(corr_unstacked.head(3))

**Naive Bayes concern:**

NB assumes feature independence. Strong correlations (e.g., age-thalach,
oldpeak-ca) violate this assumption, inflating the evidence and potentially
producing overconfident probability estimates.